In [1]:
# Import and initialize poolparty
import poolparty as pp 
pp.init()

template_pool = pp.from_seq('TCCCGACT<cre>GGAAAGCGGGCAgtGAGCACACAGG</cre>ATTACGG<bc/>AGATCGGA')\
    .named('template_pool').stylize('cre', style='uppercase red bold').stylize(which='tags',style='gray')

#template_pool.print_library()

mutated_pool = template_pool.mutagenize('cre',
                                        mutation_rate=0.1, 
                                        mode='random', 
                                        num_states=5, 
                                        mark_changes=False,
                                        changes_style='cyan bold lower',
                                        seq_name_prefix='mut_').named('mutated_pool')


mutated_pool.print_library()


mutated_pool: seq_length=48, num_states=5
state  name   seq
    0  mut_0  TCCCGACT<cre>GGAAAGCGaGCAGTGAcCAtAaAGG</cre>ATTACGG<bc/>AGATCGGA
    1  mut_1  TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGG</cre>ATTACGG<bc/>AGATCGGA
    2  mut_2  TCCCGACT<cre>GGcgAGCtGGCAtTGAGCACAgAGG</cre>ATTACGG<bc/>AGATCGGA
    3  mut_3  TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGG</cre>ATTACGG<bc/>AGATCGGA
    4  mut_4  TCCCGACT<cre>GGAAAGCGGGCAGTGAaCACcCAGG</cre>ATTACGG<bc/>AGATCGGA



Pool(id=3, name='mutated_pool', op='op[3]:mutagenize', num_states=5)

In [5]:
# Import and initialize poolparty
import poolparty as pp 
pp.init()

# # Add highlighting specifications
# pp.clear_highlights()
# pp.add_highlight(which='tags', style='gray')
# pp.add_highlight(region='cre', which='upper', style='red')
# pp.add_highlight(which='lower gap', style='bold blue')

pp.clear_pools()

template_pool = pp.from_seq('TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGG</cre>ATTACGG<bc/>AGATCGGA')\
    .named('template_pool').stylize('cre', style='salmon')

mutated_pool = template_pool.mutagenize('cre',
                                        mutation_rate=0.1, 
                                        mode='random', 
                                        num_states=5, 
                                        mark_changes=False,
                                        changes_style='yellow bold',
                                        seq_name_prefix='mut_').named('mutated_pool')

deletion_pool = template_pool.deletion_scan('cre', 
                                            deletion_length=5, 
                                            positions=slice(None, None, 5), 
                                            mode='sequential', 
                                            seq_name_prefix='del_').named('deletion_pool')

sites_pool=pp.from_seqs(['AAAAA','TTTTT'], 
                        mode='sequential', 
                        op_iter_order=-1,
                        seq_name_prefix='site_').named('sites_pool').stylize(style='cyan')

insertion_pool = template_pool.insertion_scan('cre', 
                                              ins_pool=sites_pool, 
                                              positions=slice(None,None,5), 
                                              replace=True, 
                                              mode='sequential',
                                              seq_name_pos_prefix='pos_', 
                                              seq_name_site_prefix='site_',
                                              seq_name_prefix='ins_').named('insertion_pool')

combo_pool = pp.stack([mutated_pool, deletion_pool, insertion_pool], name='stacked_pool')\
    .repeat_states(2, seq_name_prefix='v_', op_iter_order=-2, name='repeated_pool')\
    .insert_kmers('bc', mode='random', length=5, seq_name_prefix='bc_')\
    .named('combo_pool')\

#combo_pool.state.print_dag()

combo_pool.print_library(show_name=True,seed=10)

combo_pool: seq_length=None, num_states=40
state  name                          seq
    0  mut_0.v_0.bc_0                TCCCGACT<cre>GGAAAGCTGGCAGTGAGTACACAGG</cre>ATTACGG<bc>tacat</bc>AGATCGGA
    1  mut_0.v_1.bc_1                TCCCGACT<cre>GGAAAGCTGGCAGTGAGTACACAGG</cre>ATTACGG<bc>acaga</bc>AGATCGGA
    2  mut_1.v_0.bc_2                TCCCGACT<cre>TGAACGCGGGCAGTGAGCACACAGT</cre>ATTACGG<bc>gccaa</bc>AGATCGGA
    3  mut_1.v_1.bc_3                TCCCGACT<cre>TGAACGCGGGCAGTGAGCACACAGT</cre>ATTACGG<bc>atgag</bc>AGATCGGA
    4  mut_2.v_0.bc_4                TCCCGACT<cre>GGAAAGTGTGCAGTGAGTACACAGG</cre>ATTACGG<bc>cttgc</bc>AGATCGGA
    5  mut_2.v_1.bc_5                TCCCGACT<cre>GGAAAGTGTGCAGTGAGTACACAGG</cre>ATTACGG<bc>aggaa</bc>AGATCGGA
    6  mut_3.v_0.bc_6                TCCCGACT<cre>GGAAAGCGGGCAGTGAGCCCACAGG</cre>ATTACGG<bc>tcttc</bc>AGATCGGA
    7  mut_3.v_1.bc_7                TCCCGACT<cre>GGAAAGCGGGCAGTGAGCCCACAGG</cre>ATTACGG<bc>gaagt</bc>AGATCGGA
    8  mut_4.v_0.bc_8       

Pool(id=12, name='combo_pool', op='op[12]:get_kmers', num_states=40)

In [3]:
combo_pool.state.print_dag()

combo_pool.state (o=-2, n=40)
├── op[12]:get_kmers.state (o=-2, n=40)
└── repeated_pool.state (o=-2, n=40)
    └── [op=Product]
        ├── op[11]:repeat.state (o=-2, n=2)
        └── stacked_pool.state (o=-1, n=20)
            └── [op=Stack]
                ├── mutated_pool.state (o=0, n=5)
                │   └── [op=Product]
                │       ├── op[2]:mutagenize.state (o=0, n=5)
                │       └── pool[1].state (o=0, n=1)
                │           └── template_pool.state (o=0, n=1)
                ├── deletion_pool.state (o=0, n=5)
                │   └── pool[3].state (o=0, n=5)
                │       └── [op=Product]
                │           ├── op[3]:deletion_scan(marker_scan).state (o=0, n=5)
                │           └── pool[1].state (o=0, n=1)
                └── insertion_pool.state (o=-1, n=10)
                    └── [op=Product]
                        ├── pool[7].state (o=-1, n=2)
                        │   ├── op[5]:from_seqs.state (o=-1, n=2)
 

In [4]:
import statetracker as st
# Initialize manager
with st.Manager() as mgr:
    
    # Define leaf counters
    mut_state = st.State(5, name='mut_state')
    del_state = st.State(5, name='del_state')
    ins_site_state = st.State(2, name='ins_site_state')
    ins_position_state = st.State(5, name='ins_position_state')
    v_state = st.State(2).named('v_state')
    
    # Build composite counters
    ins_state = st.product([ins_position_state, ins_site_state],
        name='ins_state')
    cre_state = st.stack([mut_state, del_state, ins_state],
        name='cre_state')
    seq_state = st.product([cre_state, v_state], name='seq_state')
    
    bc_state = st.synced_to(seq_state).named('bc_state')

    # I want this 
    rows_state = st.State(6, name='rows_state')
    cols_state = st.State(8, name='cols_state')
    plate_state = st.product([rows_state, cols_state]).named('plate_state')
    
    st.sync(plate_state, seq_state)
    
# Print DAG
plate_state.print_dag('minimal')

# Print sequences
plate_state.get_iteration_df().reset_index()


plate_state (n=48)
├── seq_state (n=40)
│   └── [Product]
│       ├── cre_state (n=20)
│       │   └── [Stack]
│       │       ├── mut_state (n=5)
│       │       ├── del_state (n=5)
│       │       └── ins_state (n=10)
│       │           └── [Product]
│       │               ├── ins_position_state (n=5)
│       │               └── ins_site_state (n=2)
│       └── v_state (n=2)
├── bc_state (n=40)
└── [Product]
    ├── rows_state (n=6)
    └── cols_state (n=8)


,plate_state,seq_state,bc_state,cre_state,mut_state,del_state,ins_state,ins_position_state,ins_site_state,v_state,rows_state,cols_state
0,0,0,0,0,0,None,None,None,None,0,0,0
1,1,1,1,1,1,None,None,None,None,0,1,0
2,2,2,2,2,2,None,None,None,None,0,2,0
3,3,3,3,3,3,None,None,None,None,0,3,0
4,4,4,4,4,4,None,None,None,None,0,4,0
5,5,5,5,5,None,0,None,None,None,0,5,0
6,6,6,6,6,None,1,None,None,None,0,0,1
7,7,7,7,7,None,2,None,None,None,0,1,1
8,8,8,8,8,None,3,None,None,None,0,2,1
9,9,9,9,9,None,4,None,None,None,0,3,1
